# Ultralytics SAM 3 예측 단계별 가이드

SAM 3(Segment Anything Model 3)은 Meta의 **PCS(Promptable Concept Segmentation)** 모델이다.
기존 YOLO 계열이 *학습된 클래스만* 검출하는 것과 달리, SAM 3은
**텍스트("person", "빨간 옷 입은 사람") / 이미지 예시(bbox) / 포인트** 를 프롬프트로 받아
해당 개념에 해당하는 **모든 인스턴스**를 검출·분할·추적한다. (open-vocabulary)

## 전체 흐름

| 단계 | 내용 | 사용 클래스 |
|------|------|-------------|
| 1 | 환경 준비 (ultralytics >= 8.3.237) | - |
| 2 | 가중치 `sam3.pt` 확보 (수동 다운로드 필요) | - |
| 3 | 텍스트 프롬프트로 개념 분할 | `SAM3SemanticPredictor` |
| 4 | 결과 시각화 · 저장 | `Results.plot()` + `cv2.imwrite` |
| 5 | 이미지 예시(bbox exemplar) 프롬프트 | `SAM3SemanticPredictor` |
| 6 | SAM2 호환 point/bbox 프롬프트 | `SAM` |
| 7 | 비디오 추적 | `SAM3VideoSemanticPredictor`, `SAM3VideoPredictor` |

> 이 노트북은 같은 폴더의 `yolo_detection.py` / `yolo_segmentation.py` 와 동일한
> "모델 로드 → 예측 → `plot()` → `cv2.imwrite`" 패턴을 그대로 따른다.

## 1단계. 환경 준비

SAM 3은 **ultralytics 8.3.237 이상**에서만 동작한다. 버전이 낮으면
`ImportError: cannot import name 'SAM3SemanticPredictor'` 가 발생한다.

In [ ]:
# [1-1] ultralytics 설치 / 업그레이드
#   - 이미 최신이면 이 셀은 건너뛰어도 된다.
#   - 노트북 안에서 pip 를 쓸 때는 %pip (magic) 를 써야 현재 커널 환경에 설치된다.
%pip install -U "ultralytics>=8.3.237"

# [1-2] 버전 확인 — 8.3.237 미만이면 SAM 3 사용 불가
import ultralytics

print("ultralytics version:", ultralytics.__version__)

# [1-3] GPU 사용 가능 여부 확인
#   - SAM 3은 모델이 크다. CPU 로도 돌지만 이미지 1장에 수십 초 걸릴 수 있다.
#   - CUDA GPU 가 있으면 아래에서 device="cuda" 로 지정한다.
import torch

print("CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("사용할 device:", DEVICE)

## 2단계. 가중치 `sam3.pt` 준비 ⚠️ 중요

YOLO 가중치(`yolo26s.pt` 등)는 코드 실행 시 **자동 다운로드**되지만,
**SAM 3 가중치는 자동으로 받아지지 않는다.**

1. [Hugging Face `facebook/sam3` 페이지](https://huggingface.co/facebook/sam3)에서 **접근 권한 요청(Request access)**
2. 승인 후 `sam3.pt` 다운로드
3. 이 노트북과 같은 폴더(또는 아래 `MODEL_PATH` 가 가리키는 경로)에 저장

아래 셀은 파일이 실제로 있는지 먼저 확인해서, 없으면 안내 메시지를 띄운다.

In [ ]:
import os

# [2-1] 경로 상수 정의 — 이후 모든 셀이 이 값을 재사용한다.
MODEL_PATH = "sam3.pt"          # 다운로드한 SAM 3 가중치
IMAGE_PATH = "images/seg.png"   # 예측 대상 이미지 (원하는 파일로 교체)
OUTPUT_DIR = "images"           # 결과 저장 폴더

# [2-2] 가중치 존재 여부 확인
if os.path.exists(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / 1024 / 1024
    print(f"OK 가중치 확인: {MODEL_PATH} ({size_mb:.1f} MB)")
else:
    print(f"[!] {MODEL_PATH} 없음.")
    print("    https://huggingface.co/facebook/sam3 에서 접근 요청 후 내려받아 이 폴더에 두세요.")

# [2-3] 입력 이미지 존재 여부 확인
print(f"입력 이미지: {IMAGE_PATH} — 존재: {os.path.exists(IMAGE_PATH)}")

## 3단계. 텍스트 프롬프트로 개념 분할 (핵심 기능)

SAM 3의 대표 기능. **찾고 싶은 것을 글자로 적으면** 그 개념에 해당하는 모든 객체를 찾아준다.

### `overrides` 딕셔너리 항목 설명

| 키 | 의미 |
|----|------|
| `conf` | 신뢰도 임계값. 낮추면 더 많이 검출(오탐 증가), 높이면 확실한 것만 |
| `task` | `"segment"` — SAM 3은 분할 모델이므로 고정 |
| `mode` | `"predict"` — 추론 모드 |
| `model` | 가중치 경로 |
| `quantize` | 가중치 정밀도. `16`(FP16)이면 VRAM 절약 + 속도 향상. CPU에서는 `32` 권장 |
| `save` | `True` 면 `runs/segment/predict*/` 에 결과 이미지 자동 저장 |

### 호출 패턴 (YOLO와 다른 점)

`model(image)` 한 방에 끝나는 YOLO와 달리 **2단계**로 나뉜다.

```python
predictor.set_image("이미지")     # 1) 이미지 인코딩 (무거움, 1번만)
results = predictor(text=[...])  # 2) 프롬프트 질의 (가벼움, 여러 번 반복 가능)
```

이미지 인코딩 결과가 캐시되므로, **같은 이미지에 프롬프트만 바꿔 여러 번 물어볼 때 훨씬 빠르다.**

In [ ]:
from ultralytics.models.sam import SAM3SemanticPredictor

# [3-1] 예측기 설정값 구성
overrides = dict(
    conf=0.25,          # 신뢰도 임계값 (0.0 ~ 1.0)
    task="segment",     # SAM 3은 분할 태스크
    mode="predict",     # 추론 모드
    model=MODEL_PATH,   # 2단계에서 확인한 가중치
    quantize=16,        # FP16 — CPU 환경이면 32 로 바꾸세요
    save=False,         # runs/ 자동 저장 끔 (아래 4단계에서 직접 저장)
    device=DEVICE,      # "cuda" 또는 "cpu"
)

# [3-2] 예측기 생성 — 이 시점에 가중치가 메모리로 올라간다 (수십 초 소요 가능)
predictor = SAM3SemanticPredictor(overrides=overrides)

# [3-3] 이미지 인코딩 — 무거운 연산. 이미지 1장당 1번만 하면 된다.
predictor.set_image(IMAGE_PATH)

# [3-4] 텍스트 프롬프트로 질의
#   - 리스트의 각 문자열이 하나의 "클래스"가 된다 (index 0 = "person", 1 = "bus", ...)
#   - 이미지에 없는 개념을 넣어도 에러 없이 결과 0개로 나온다.
results = predictor(text=["person", "bus", "glasses"])

# [3-5] 검출 개수 확인
#   - results 는 리스트. results[0] 이 첫 이미지의 Results 객체.
print(f"검출된 객체 수: {len(results[0].boxes)}")

In [ ]:
# [3-6] 서술형 프롬프트 — SAM 3의 진짜 강점
#   단순 명사뿐 아니라 속성을 붙인 구(phrase)도 구분해낸다.
#   set_image() 를 다시 호출할 필요 없음 → 인코딩 캐시 재사용 → 빠름
results_desc = predictor(text=["person with red cloth", "person with blue cloth"])

print(f"서술형 프롬프트 검출 수: {len(results_desc[0].boxes)}")

# [3-7] 결과 내부 구조 살펴보기
r = results_desc[0]
print("클래스 이름 매핑:", r.names)        # {0: 'person with red cloth', 1: '...'}
print("bbox 텐서 shape :", None if r.boxes is None else r.boxes.xyxy.shape)
print("mask 텐서 shape :", None if r.masks is None else r.masks.data.shape)

# [3-8] 객체별 상세 출력
for i, box in enumerate(r.boxes):
    cls_id = int(box.cls[0])              # 클래스 인덱스
    conf = float(box.conf[0])             # 신뢰도
    xyxy = box.xyxy[0].tolist()           # [x1, y1, x2, y2] 픽셀 좌표
    print(f"  #{i} {r.names[cls_id]:<25} conf={conf:.3f} bbox={[round(v, 1) for v in xyxy]}")

## 4단계. 결과 시각화 · 저장

`yolo_segmentation.py` 와 **완전히 동일한 패턴**이다. SAM 3의 결과도 결국
일반 `Results` 객체라서 `plot()` 으로 마스크·박스가 그려진 BGR numpy 배열을 얻는다.

> `plot()` 반환값은 **BGR** 순서다. `cv2.imwrite` 는 BGR을 기대하므로 그대로 넣으면 되지만,
> `matplotlib` 로 표시할 때는 RGB로 뒤집어야 색이 정상으로 보인다.

In [ ]:
import cv2

# [4-1] 첫 번째 결과를 이미지로 렌더링 (BGR numpy 배열 반환)
res_img = results[0].plot()

# [4-2] 원본 파일명에서 확장자 떼기 — "images/seg.png" -> "seg"
base_name = os.path.splitext(os.path.basename(IMAGE_PATH))[0]

# [4-3] 저장 경로 조립 후 기록
output_image_path = f"{OUTPUT_DIR}/{base_name}_sam3.jpg"
cv2.imwrite(output_image_path, res_img)
print(f"예측 결과 이미지가 잘 저장 되었습니다. {output_image_path}")

# [4-4] 노트북 안에서 바로 확인 (BGR -> RGB 변환 필수)
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(res_img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("SAM 3 — text prompt segmentation")
plt.show()

## 5단계. 이미지 예시(exemplar) 프롬프트

**말로 설명하기 애매한 대상**일 때 사용한다. 찾고 싶은 물체 하나를 bbox로 찍어주면
SAM 3이 "이것과 같은 종류"를 이미지 전체에서 찾아낸다.

- `bboxes` 형식: `[[x1, y1, x2, y2], ...]` — **픽셀 좌표**, 정규화 값 아님
- 여러 개를 주면 예시가 늘어나 정확도가 올라간다
- 텍스트와 조합해서 함께 넘길 수도 있다

In [ ]:
# [5-1] 예시 bbox 1개 — "이 박스 안의 물체와 같은 것들을 찾아라"
#   좌표는 실제 이미지 크기에 맞게 바꿔야 한다.
results_ex = predictor(bboxes=[[480.0, 290.0, 590.0, 650.0]])
print(f"예시 1개 기준 검출 수: {len(results_ex[0].boxes)}")

# [5-2] 예시 bbox 여러 개 — 예시가 많을수록 개념이 뚜렷해진다
results_ex2 = predictor(bboxes=[[539, 599, 589, 639], [343, 267, 499, 662]])
print(f"예시 2개 기준 검출 수: {len(results_ex2[0].boxes)}")

# [5-3] 저장
cv2.imwrite(f"{OUTPUT_DIR}/{base_name}_sam3_exemplar.jpg", results_ex[0].plot())
print(f"저장 완료: {OUTPUT_DIR}/{base_name}_sam3_exemplar.jpg")

## 6단계. SAM 2 호환 방식 — 포인트 / 박스 프롬프트

"개념 전체"가 아니라 **특정 위치 하나**만 분할하고 싶을 때는
기존 `SAM` 클래스를 그대로 쓰면 된다. 호출법이 YOLO와 거의 같아서 가장 간단하다.

- `points=[x, y]` + `labels=[1]` → 1 = 전경(이걸 포함), 0 = 배경(이건 제외)
- `bboxes=[x1, y1, x2, y2]` → 이 박스 안의 물체 하나를 분할

5단계의 exemplar와 헷갈리기 쉬운데, 목적이 다르다.

| | 5단계 exemplar | 6단계 SAM 호환 |
|---|---|---|
| 입력 bbox 의미 | "이런 종류를 다 찾아줘" | "딱 이 박스 안의 것만 분할해줘" |
| 출력 개수 | 여러 개 (전체 인스턴스) | 보통 1개 |

In [ ]:
from ultralytics import SAM

# [6-1] SAM 래퍼로 로드 — YOLO("yolo26s.pt") 와 동일한 사용감
model = SAM(MODEL_PATH)

# [6-2] 모델 정보 출력 (선택)
model.info()

# [6-3] 포인트 프롬프트 — (900, 370) 지점에 있는 물체를 분할
#   labels=[1] -> 전경. 0 을 주면 "여기는 빼라"는 뜻.
res_point = model.predict(source=IMAGE_PATH, points=[900, 370], labels=[1])
cv2.imwrite(f"{OUTPUT_DIR}/{base_name}_sam3_point.jpg", res_point[0].plot())

# [6-4] 박스 프롬프트 — 박스 안 물체 하나를 분할
res_box = model.predict(source=IMAGE_PATH, bboxes=[100, 150, 300, 400])
cv2.imwrite(f"{OUTPUT_DIR}/{base_name}_sam3_box.jpg", res_box[0].plot())

print("포인트/박스 프롬프트 결과 저장 완료")

## 7단계. 비디오 추적

SAM 3은 영상에서 **동일 객체에 같은 ID를 유지하며** 추적한다.

| 클래스 | 프롬프트 | 용도 |
|--------|----------|------|
| `SAM3VideoSemanticPredictor` | `text=[...]` | "영상 속 모든 사람/자전거 추적" |
| `SAM3VideoPredictor` | `bboxes=[...]` | "첫 프레임의 이 물체만 계속 따라가기" |

> **`stream=True` 를 반드시 쓸 것.** 생략하면 전체 프레임 결과를 한꺼번에 메모리에 쌓아
> 긴 영상에서 OOM(메모리 부족)이 난다. `stream=True` 면 제너레이터로 프레임마다 흘려보낸다.

In [ ]:
from ultralytics.models.sam import SAM3VideoSemanticPredictor

VIDEO_PATH = "images/sample.mp4"  # 실제 영상 경로로 교체

if os.path.exists(VIDEO_PATH):
    # [7-1] 비디오용 설정 — imgsz 를 낮추면 속도가 크게 빨라진다
    v_overrides = dict(
        conf=0.25,
        task="segment",
        mode="predict",
        imgsz=640,
        model=MODEL_PATH,
        quantize=16,
        save=True,      # runs/segment/predict*/ 에 주석 달린 영상 저장
        device=DEVICE,
    )
    v_predictor = SAM3VideoSemanticPredictor(overrides=v_overrides)

    # [7-2] 텍스트로 추적 — stream=True 이므로 아직 계산 안 됨 (제너레이터)
    v_results = v_predictor(source=VIDEO_PATH, text=["person", "bicycle"], stream=True)

    # [7-3] 프레임 단위로 소비해야 실제 추론이 돈다
    for frame_idx, r in enumerate(v_results):
        n = 0 if r.boxes is None else len(r.boxes)
        print(f"frame {frame_idx:04d}: {n} objects")
        if frame_idx >= 9:  # 데모용으로 10프레임만
            break
else:
    print(f"'{VIDEO_PATH}' 없음 — 비디오 단계 건너뜀.")

In [ ]:
from ultralytics.models.sam import SAM3VideoPredictor

if os.path.exists(VIDEO_PATH):
    # [7-4] 시각 프롬프트 추적 — 첫 프레임에서 지정한 박스의 객체만 끝까지 따라간다
    vp_overrides = dict(
        conf=0.25, task="segment", mode="predict",
        model=MODEL_PATH, quantize=16, device=DEVICE,
    )
    vp = SAM3VideoPredictor(overrides=vp_overrides)

    vp_results = vp(source=VIDEO_PATH, bboxes=[[706.5, 442.5, 905.25, 555]], stream=True)

    for frame_idx, r in enumerate(vp_results):
        if frame_idx >= 9:
            break
        print(f"frame {frame_idx:04d} 추적 중")
else:
    print(f"'{VIDEO_PATH}' 없음 — 건너뜀.")

## 정리 · 트러블슈팅

### 클래스 선택 요약

| 하고 싶은 것 | 클래스 | 호출 |
|--------------|--------|------|
| 텍스트로 개념 전체 찾기 | `SAM3SemanticPredictor` | `set_image()` 후 `predictor(text=[...])` |
| 예시 이미지로 같은 종류 찾기 | `SAM3SemanticPredictor` | `predictor(bboxes=[[...]])` |
| 특정 위치 하나만 분할 | `SAM` | `model.predict(points=..., labels=...)` |
| 영상에서 개념 추적 | `SAM3VideoSemanticPredictor` | `(source=..., text=[...], stream=True)` |
| 영상에서 지정 객체 추적 | `SAM3VideoPredictor` | `(source=..., bboxes=[[...]], stream=True)` |

### 자주 만나는 문제

| 증상 | 원인 / 해결 |
|------|-------------|
| `cannot import name 'SAM3SemanticPredictor'` | ultralytics 버전 낮음. `pip install -U "ultralytics>=8.3.237"` |
| `FileNotFoundError: sam3.pt` | 가중치 자동 다운로드 안 됨. HF에서 접근 요청 후 수동 배치 (2단계) |
| `CUDA out of memory` | `quantize=16` 유지 + `imgsz` 낮추기 + 비디오는 `stream=True` |
| CPU에서 너무 느림 | 정상. SAM 3은 대형 모델이므로 GPU 권장, 또는 `imgsz` 축소 |
| 검출이 하나도 안 됨 | `conf` 를 0.1 정도로 낮추고, 프롬프트 단어를 더 일반적인 표현으로 |
| 저장한 이미지 색이 이상함 | `plot()` 은 BGR. `plt.imshow` 전에 `cv2.cvtColor(..., COLOR_BGR2RGB)` |

### YOLO 대비 언제 SAM 3을 쓰나

- **YOLO(`yolo26s-seg.pt`)**: 미리 정해진 클래스, 실시간 속도 필요. 대부분의 실전 서비스
- **SAM 3**: 학습 안 된 새 개념을 즉시 찾아야 할 때, 라벨링 자동화, 프로토타이핑